# 01 — Molecular structure as a low-data learning problem

**Scientific thread**

We start from the physical object, not from the algorithm:

\[
(T,P) \longrightarrow g_{\mathrm{OO}}(r).
\]

**Goal:** understand what one training example means, inspect how molecular structure changes across thermodynamic space, and see why validation consumes part of a small information budget.

> **Workshop rhythm:** run the short experiment cells, inspect the figure, then discuss the questions before moving on.

In [ ]:
#@title 0. Workshop setup — run once { display-mode: "form" }
# Infrastructure is intentionally hidden so workshop time stays focused on physics.

from pathlib import Path
import hashlib, importlib.util, os, shutil, subprocess, sys, time, urllib.request, zipfile

ASSET_URL = "https://github.com/Soft-Condensed-Matter/ThermoRDF-LowData-Workshop/releases/download/student-colab-v1.0-rc1/ThermoRDF-Colab-Assets.zip"
EXPECTED_ASSET_SHA256 = "2e75fd65ad39a9dec41f7b089c2aafe51a6bb14eeee049b055be8ea3953a7bea"
EXPECTED_ASSET_VERSION = "ThermoRDF Colab Assets v1.0"
WORKSHOP_ROOT = Path("/content/ThermoRDF-Workshop")
ASSET_NAME = "ThermoRDF-Colab-Assets.zip"

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _download(url, destination, attempts=3):
    tmp = destination.with_suffix(destination.suffix + ".part")
    if tmp.exists():
        tmp.unlink()
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "ThermoRDF-LowData-Workshop/1.0"}
            )
            with urllib.request.urlopen(request, timeout=90) as response, open(tmp, "wb") as fh:
                shutil.copyfileobj(response, fh)
            tmp.replace(destination)
            return
        except Exception as exc:
            last_error = exc
            if tmp.exists():
                tmp.unlink()
            if attempt < attempts:
                time.sleep(2 * attempt)
    raise RuntimeError(
        "Could not download the workshop assets from GitHub. "
        "Check the internet connection and rerun this cell."
    ) from last_error

def _assets_ready(root):
    required = [
        root / "COLAB_ASSET_VERSION.txt",
        root / "data/metadata/stage_02_radial_grid.csv",
        root / "data/models/stage_07_selected_b48.pt",
        root / "data/teaching/stage_02_b48_train_40.csv.gz",
        root / "data/teaching/stage_02_b48_validation_8.csv.gz",
        root / "data/teaching/stage_02_oo_reference_372.csv.gz",
        root / "src/thermordf_workshop/__init__.py",
    ]
    if not all(path.is_file() for path in required):
        return False
    return (root / "COLAB_ASSET_VERSION.txt").read_text().strip() == EXPECTED_ASSET_VERSION

# Local/instructor override used only for automated validation.
_local_root = os.environ.get("THERMORDF_WORKSHOP_ROOT", "").strip()
if _local_root:
    WORKSHOP_ROOT = Path(_local_root).resolve()
else:
    if not _assets_ready(WORKSHOP_ROOT):
        archive = Path("/content") / ASSET_NAME

        # Reuse a verified archive if this runtime already downloaded it.
        if archive.is_file() and _sha256(archive) != EXPECTED_ASSET_SHA256:
            archive.unlink()

        if not archive.is_file():
            print("Downloading workshop assets ...")
            _download(ASSET_URL, archive)

        digest = _sha256(archive)
        if digest != EXPECTED_ASSET_SHA256:
            archive.unlink(missing_ok=True)
            raise RuntimeError(
                "Workshop asset checksum mismatch. "
                "Please rerun this cell to download a clean copy."
            )

        if WORKSHOP_ROOT.exists():
            shutil.rmtree(WORKSHOP_ROOT)
        WORKSHOP_ROOT.mkdir(parents=True)

        with zipfile.ZipFile(archive) as zf:
            zf.extractall(WORKSHOP_ROOT)

        if not _assets_ready(WORKSHOP_ROOT):
            raise RuntimeError(
                "Workshop assets were downloaded but the extracted bundle is incomplete."
            )

        print("✓ Workshop assets downloaded and verified")

_required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "torch": "torch",
}
_missing = [pkg for pkg, module in _required.items() if importlib.util.find_spec(module) is None]
if _missing:
    print("Installing missing Colab packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

src_path = str(WORKSHOP_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from thermordf_workshop import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.set_num_threads(min(2, os.cpu_count() or 1))
data = load_workshop_data(WORKSHOP_ROOT)

print(
    f"✓ Workshop ready | {len(data.train)} training + "
    f"{len(data.validation)} validation states | "
    f"{len(data.r_nm)} RDF coordinates"
)

## Physical question — what exactly is scarce?

A thermodynamic state gives us **one structured molecular observation**: an RDF sampled at 725 radial coordinates.

Those 725 values are strongly correlated parts of the same curve. They are **not 725 independent training examples**.

In [ ]:
print(f"training states   : {len(data.train)}")
print(f"validation states : {len(data.validation)}")
print(f"RDF coordinates   : {len(data.r_nm)}")

## Experiment 1 — look at the structure before learning it

The curves below come only from the development training set. Focus on the first peak, first minimum, subsequent oscillations and relaxation towards the bulk value.

In [ ]:
plot_representative_rdfs(data);

### Observe

- Which RDF features change most visibly with thermodynamic state?
- Which features mostly change in amplitude, and which shift in position?
- Which radial regions look smooth enough to be learnable from nearby states?
- Why is predicting the **entire curve** harder than predicting a single scalar?

## Experiment 2 — allocate a small simulation budget

The thermodynamic domain is broad, but the development budget contains only **48 molecular states**. Forty states provide fitting information and eight are withheld for development validation.

In [ ]:
plot_development_budget(data);

In [ ]:
data.validation[["state_id", "T_K", "P_bar"]].reset_index(drop=True)

### Interpret

A held-out state is **not free information**. Every validation state is one molecular simulation that cannot simultaneously be used to fit the model.

\[
\boxed{\text{learn a thermodynamic–structural map from deliberately few states}}
\]

**Take-away:** the low-data condition is defined by the number and placement of independent thermodynamic states—not by the 725 coordinates used to represent each RDF.